# Diamond Photonic Crystal Nanocavity
## FDTD Simulation and Optical Characterisation with Tidy3D

[![Tidy3D](https://img.shields.io/badge/Tidy3D-2.x-blue)](https://docs.flexcompute.com/projects/tidy3d/en/latest/)  [![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-green)](https://python.org)  [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

This notebook presents a complete FDTD simulation workflow for characterising a **photonic crystal nanocavity** fabricated in a diamond thin film. The device is designed to confine light and enhance the spontaneous emission of **tin-vacancy (SnV) colour centres** near $\lambda \approx 615$–$645$ nm through the **Purcell effect**.

It is best read as the detailed single-cavity characterisation companion to the broader fabrication-variation study of diamond photonic crystal cavities reported in [arXiv:2601.20025](https://arxiv.org/abs/2601.20025): here the geometry is fixed by the fabricated GDS, and the emphasis is on modal analysis rather than parameter sweeps.

### What you will learn

- Build a photonic crystal cavity in Tidy3D directly from GDS layout files
- Run a **two-stage simulation**: broadband resonance search → narrowband full characterisation
- Extract the Q-factor from the FDTD ringdown using the Tidy3D **ResonanceFinder**
- Compute mode volume, Purcell factor, collection efficiency, and far-field polarisation

### Device at a glance

| Parameter | Value |
|-----------|-------|
| Material | Diamond (n ≈ 2.41 @ 640 nm) |
| Slab thickness | 136 nm |
| Target wavelength | ~620–640 nm (SnV zero-phonon line region) |
| Sidewall angle | 15.6° (fabrication taper) |
| Cavity type | Nanobeam photonic crystal |

## Background

Photonic crystal (PhC) nanocavities in diamond are a key platform for **solid-state quantum photonics**: by simultaneously confining light to sub-cubic-wavelength volumes and maintaining high quality factors, they dramatically accelerate the spontaneous emission rate of embedded colour centres via the **Purcell effect**:

$$F_P = \frac{3}{4\pi^2} \left(\frac{\lambda_0}{n}\right)^3 \frac{Q}{V_{\text{eff}}}$$

where $Q$ is the quality factor, $V_{\text{eff}}$ the effective mode volume, $\lambda_0$ the free-space resonance wavelength, and $n$ the refractive index at the emitter site.

Within the context of [arXiv:2601.20025](https://arxiv.org/abs/2601.20025), this notebook should be viewed as the high-detail analysis of one fabricated cavity instance: thickness, sidewall angle, and etched hole pattern are taken as given, and we inspect how that realised geometry supports the optical mode.

### Simulation workflow

We use a **two-stage FDTD approach** to balance computational cost with accuracy:

| Stage | Bandwidth | Monitors | Purpose |
|-------|-----------|----------|---------|
| **Scout** | 12 % (broadband) | point probe | Locate resonance; extract Q and decay time |
| **Lock-in** | 2 % (narrowband) | near-field, 3-D field, far-field | Mode volume, collection efficiency, polarisation |

The scout stage uses a short, broad Gaussian pulse to excite all cavity modes in the spectral window; the time-domain ringdown is decomposed into decaying sinusoids using Prony's method (Tidy3D `ResonanceFinder`). The detected resonance wavelength then seeds the lock-in stage, which resolves the cavity mode in detail.

## Table of Contents

1. [Imports and Configuration](#imports)
2. [Material Properties – Diamond](#material)
3. [Photonic Crystal Cavity Geometry](#geometry)
4. [Stage 1 – Broadband Resonance Search](#scout)
5. [Q-Factor Extraction](#qfactor)
6. [Stage 2 – Narrowband Lock-in Simulation](#lockin)
7. [Near-Field Mode Analysis](#nearfield)
8. [Mode Volume and Purcell Enhancement](#modevolume)
9. [Far-Field Radiation and Collection Efficiency](#farfield)
10. [Polarisation Analysis](#polarisation)
11. [Results Summary](#summary)

<a id='imports'></a>
## 1. Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler
from scipy.optimize import curve_fit
from scipy.signal import hilbert
import warnings
import json
from pathlib import Path

warnings.filterwarnings('ignore')

import tidy3d as td
from tidy3d.plugins.resonance import ResonanceFinder
import gdstk

print(f'tidy3d version: {td.__version__}')

In [ ]:
# ── Physical parameters ───────────────────────────────────────────────────────
THICKNESS_UM       = 0.136        # Diamond slab thickness [µm]
WAVELENGTH_SCOUT   = 0.630        # Broadband centre wavelength [µm]  (630 nm)
SIDEWALL_ANGLE_DEG = 15.6         # Fabrication sidewall angle [degrees]
NA                 = 0.65         # Collection numerical aperture
N_BG               = 1.0          # Background refractive index (air)
C0                 = 299_792_458.0  # Speed of light [m/s]

# ── GDS layout files (relative to this notebook) ─────────────────────────────
CAVITY_GDS = Path('../gds/Cavity_Fab.gds')
HOLES_GDS  = Path('../gds/Holes_Fab.gds')

# ── Cached analysis-grade results ────────────────────────────────────────────
SCOUT_RESULTS  = Path('../data/results/results_scout_q_only_0.136um.hdf5')
LOCKIN_RESULTS = Path('../data/results/results_lockin_full_0.136um.hdf5')

# ── Simulation profile ───────────────────────────────────────────────────────
# `analysis_full` keeps the original notebook fidelity and reruns both stages.
# `sobol_cheap` follows the low-cost Sobol scout philosophy: 620 nm centre,
# 7 ps runtime, 13 steps/wavelength, and it skips rerunning the expensive
# lock-in stage unless you force it back on below.
SIMULATION_PROFILE = 'analysis_full'  # 'analysis_full' or 'sobol_cheap'
FORCE_RUN_LOCKIN_STAGE = False

PROFILE_CONFIG = {
    'analysis_full': dict(
        scout_wavelength_um=0.630,
        min_steps_per_wvl=18,
        scout_run_time_ps=12.0,
        lockin_run_time_ps=8.0,
        scout_bandwidth_rel=0.12,
        lockin_bandwidth_rel=0.02,
        run_lockin_stage=True,
    ),
    'sobol_cheap': dict(
        scout_wavelength_um=0.620,
        min_steps_per_wvl=13,
        scout_run_time_ps=7.0,
        lockin_run_time_ps=8.0,
        scout_bandwidth_rel=0.12,
        lockin_bandwidth_rel=0.02,
        run_lockin_stage=False,
    ),
}

if SIMULATION_PROFILE not in PROFILE_CONFIG:
    raise ValueError(
        f'Unknown SIMULATION_PROFILE={SIMULATION_PROFILE!r}. '
        f'Choose from {tuple(PROFILE_CONFIG)}.'
    )

PROFILE = PROFILE_CONFIG[SIMULATION_PROFILE]
SCOUT_WAVELENGTH_UM = PROFILE['scout_wavelength_um']
MIN_STEPS_PER_WVL   = PROFILE['min_steps_per_wvl']
SCOUT_RUN_TIME_PS   = PROFILE['scout_run_time_ps']
LOCKIN_RUN_TIME_PS  = PROFILE['lockin_run_time_ps']
SCOUT_BANDWIDTH     = PROFILE['scout_bandwidth_rel']
LOCKIN_BANDWIDTH    = PROFILE['lockin_bandwidth_rel']
RUN_LOCKIN_STAGE    = PROFILE['run_lockin_stage'] or FORCE_RUN_LOCKIN_STAGE

# Keep cheap reruns separate so the canonical analysis cache is preserved.
SCOUT_RUN_RESULTS = (
    SCOUT_RESULTS
    if SIMULATION_PROFILE == 'analysis_full'
    else Path('../data/results/results_scout_sobol_like_0.136um.hdf5')
)
LOCKIN_RUN_RESULTS = (
    LOCKIN_RESULTS
    if SIMULATION_PROFILE == 'analysis_full'
    else Path('../data/results/results_lockin_sobol_like_0.136um.hdf5')
)
SCOUT_LOAD_RESULTS  = SCOUT_RUN_RESULTS if SCOUT_RUN_RESULTS.exists() else SCOUT_RESULTS
LOCKIN_LOAD_RESULTS = LOCKIN_RUN_RESULTS if LOCKIN_RUN_RESULTS.exists() else LOCKIN_RESULTS
SCOUT_TASK_NAME = f'diamond_cavity_scout_{SIMULATION_PROFILE}'
LOCKIN_TASK_NAME = f'diamond_cavity_lockin_{SIMULATION_PROFILE}'

# ── Simulation domain parameters ─────────────────────────────────────────────
# The nanobeam GDS spans ~31 µm; only the central 21 µm is simulated —
# 5 µm is trimmed from each x-end (mirror terminations) to reduce cost.
# y and z are padded with 2 µm PML buffers on each side.
X_TRIM_UM  = 5.0   # µm trimmed from each long-axis end of the GDS
PML_PAD_UM = 2.0   # µm PML buffer on y/z faces
PML_LAYERS = 12    # PML absorption layers (12 for high-Q accuracy)

# ── Execution flag ────────────────────────────────────────────────────────────
# When True, the selected profile is submitted to Tidy3D cloud.
# `sobol_cheap` reruns only Stage 1 by default and reuses the lock-in cache.
# When False, the notebook loads the available cached HDF5 results.
RUN_SIMULATIONS = False

In [ ]:
# ── Publication-style plot theme ──────────────────────────────────────────────
RED    = '#840032'
BLUE   = '#002642'
YELLOW = '#e59500'
WHITE  = '#FFFAF2'
BLACK  = '#02040f'
PALETTE = [RED, BLUE, YELLOW, BLACK]

# Monopolar colormap: black → blue → red → yellow → white
mono_cmap = LinearSegmentedColormap.from_list(
    'mono', [BLACK, BLUE, RED, YELLOW, WHITE], N=256
)
# Bipolar colormap: black → blue → white → yellow → red
bipolar_cmap = LinearSegmentedColormap.from_list(
    'bipolar', [BLACK, BLUE, WHITE, YELLOW, RED], N=256
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['axes.prop_cycle'] = cycler(color=PALETTE)
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.size'] = 11

print('Plot theme ready.')

<a id='material'></a>
## 2. Material Properties – Diamond

Diamond's refractive index is described by a two-term **Sellmeier equation** (Zaitsev 2001):

$$n^2(\lambda) = 1 + \frac{B_1 \lambda^2}{\lambda^2 - C_1} + \frac{B_2 \lambda^2}{\lambda^2 - C_2}$$

with $B_1 = 0.3306$, $C_1 = 0.175^2\,\mu\text{m}^2$, $B_2 = 4.3356$, $C_2 = 0.106^2\,\mu\text{m}^2$. The model is valid from 0.23 to 5 µm.

In [ ]:
def n_diamond(wavelength_um):
    """
    Diamond refractive index via the two-term Sellmeier equation.

    Reference: Zaitsev, Optical Properties of Diamond (2001).
    Valid range: 0.23 – 5 µm.

    Parameters
    ----------
    wavelength_um : float  — free-space wavelength [µm]

    Returns
    -------
    float  — refractive index at the given wavelength
    """
    lam2 = np.asarray(wavelength_um, dtype=float) ** 2
    B1, C1 = 0.3306, 0.175 ** 2   # first oscillator
    B2, C2 = 4.3356, 0.106 ** 2   # second oscillator
    n2 = 1.0 + B1 * lam2 / (lam2 - C1) + B2 * lam2 / (lam2 - C2)
    return np.sqrt(n2)


# Tidy3D dispersive medium (Sellmeier coefficients)
diamond_medium = td.Sellmeier(coeffs=[(0.3306, 0.175 ** 2), (4.3356, 0.106 ** 2)])
air_medium     = td.Medium(permittivity=1.0)

n_scout = float(n_diamond(SCOUT_WAVELENGTH_UM))
print(f'n_diamond at {SCOUT_WAVELENGTH_UM * 1e3:.0f} nm : {n_scout:.4f}')

# ── Dispersion curve ──────────────────────────────────────────────────────────
wl   = np.linspace(0.45, 1.0, 400)
n_wl = n_diamond(wl)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(wl * 1e3, n_wl, color=RED, lw=2, label='Diamond')
ax.axvline(SCOUT_WAVELENGTH_UM * 1e3, color=BLUE, ls='--', lw=1.5,
           label=f'{SCOUT_WAVELENGTH_UM * 1e3:.0f} nm  (n = {n_scout:.3f})')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Refractive index $n$')
ax.set_title('Diamond Dispersion (Sellmeier)')
ax.legend(framealpha=0.85)
ax.set_xlim(450, 1000)
plt.tight_layout()
plt.show()

<a id='geometry'></a>
## 3. Photonic Crystal Cavity Geometry

The cavity geometry is imported from two GDS files:

- **`Cavity_Fab.gds`** — the diamond nanobeam outline (rectangle or tapered polygon)
- **`Holes_Fab.gds`** — the photonic crystal hole array (ellipses/circles, layer 0)

The slab is extruded with the measured fabrication **sidewall angle of 15.6°**, producing a trapezoidal cross-section that accurately models the reactive-ion-etching profile. Holes are punched through the slab as air voids.

> **Note:** All coordinates are in micrometres (µm); Tidy3D uses µm as its default length unit.

In [ ]:
def load_cavity_geometry(cavity_gds_path, holes_gds_path,
                         thickness_um, sidewall_angle_deg,
                         cavity_medium, holes_medium,
                         hole_layer=0, hole_dtype=0,
                         chunk_size=80):
    """
    Build Tidy3D structures from GDS layout files.

    The cavity slab uses `td.PolySlab` with a sidewall_angle to model the
    fabrication taper.  Holes are loaded from the GDS polygons on the
    specified layer/datatype and grouped in chunks for solver efficiency.

    Parameters
    ----------
    cavity_gds_path, holes_gds_path : Path
    thickness_um : float
    sidewall_angle_deg : float
    cavity_medium, holes_medium : td.Medium
    hole_layer, hole_dtype : int
    chunk_size : int  — max geometries per Structure group

    Returns
    -------
    structures : list[td.Structure]
    bbox : (xmin, xmax, ymin, ymax)  in µm
    """
    angle_rad = np.deg2rad(sidewall_angle_deg)
    z_bounds  = (-thickness_um / 2, thickness_um / 2)

    # ── Cavity slab ───────────────────────────────────────────────────────────
    lib_cav   = gdstk.read_gds(str(cavity_gds_path))
    scale_cav = lib_cav.unit / 1e-6        # → µm
    cell_cav  = lib_cav.top_level()[0]
    bb        = cell_cav.bounding_box()
    xmin_c    = bb[0][0] * scale_cav
    ymin_c    = bb[0][1] * scale_cav
    xmax_c    = bb[1][0] * scale_cav
    ymax_c    = bb[1][1] * scale_cav

    verts_cav = [
        (xmin_c, ymin_c), (xmin_c, ymax_c),
        (xmax_c, ymax_c), (xmax_c, ymin_c),
    ]
    core_slab = td.Structure(
        geometry=td.PolySlab(
            vertices=verts_cav, axis=2,
            slab_bounds=z_bounds,
            reference_plane='middle',
            sidewall_angle=angle_rad,
        ),
        medium=cavity_medium,
    )

    # ── Photonic crystal holes ─────────────────────────────────────────────────
    lib_h     = gdstk.read_gds(str(holes_gds_path))
    scale_h   = lib_h.unit / 1e-6
    cell_h    = lib_h.top_level()[0]

    polys     = [p for p in cell_h.polygons
                 if p.layer == hole_layer and p.datatype == hole_dtype]

    hole_structs = []
    for i in range(0, len(polys), chunk_size):
        geoms = [
            td.PolySlab(
                vertices=[(pt[0] * scale_h, pt[1] * scale_h) for pt in p.points],
                axis=2, slab_bounds=z_bounds, reference_plane='middle',
                sidewall_angle=angle_rad,
            )
            for p in polys[i : i + chunk_size]
        ]
        geo = td.GeometryGroup(geometries=geoms) if len(geoms) > 1 else geoms[0]
        hole_structs.append(td.Structure(geometry=geo, medium=holes_medium))

    structures = [core_slab] + hole_structs
    print(f'Cavity outline : x=[{xmin_c:.3f}, {xmax_c:.3f}] µm,'
          f'  y=[{ymin_c:.3f}, {ymax_c:.3f}] µm')
    print(f'Holes loaded   : {len(polys)} polygons → {len(hole_structs)} structure groups')

    return structures, (xmin_c, xmax_c, ymin_c, ymax_c)


structures, (X0, X1, Y0, Y1) = load_cavity_geometry(
    CAVITY_GDS, HOLES_GDS,
    THICKNESS_UM, SIDEWALL_ANGLE_DEG,
    diamond_medium, air_medium,
)

CX = (X0 + X1) / 2   # cavity centre x [µm]
CY = (Y0 + Y1) / 2   # cavity centre y [µm]
print(f'Cavity centre  : ({CX:.4f}, {CY:.4f}) µm')

In [ ]:
# ── Top-view geometry preview ─────────────────────────────────────────────────
lib_vis   = gdstk.read_gds(str(HOLES_GDS))
scale_vis = lib_vis.unit / 1e-6
cell_vis  = lib_vis.top_level()[0]

fig, ax = plt.subplots(figsize=(13.2, 4.8))

# Diamond slab background
rect = plt.Rectangle((X0, Y0), X1 - X0, Y1 - Y0,
                      fc=BLUE, ec='none', alpha=0.85, label='Diamond slab')
ax.add_patch(rect)

# Photonic crystal holes
for poly in cell_vis.polygons:
    if poly.layer == 0 and poly.datatype == 0:
        pts  = poly.points * scale_vis
        patch = plt.Polygon(pts, closed=True, fc=WHITE, ec=BLACK, lw=0.4)
        ax.add_patch(patch)

# Dipole source position
ax.plot(CX, CY, 'x', color=RED, ms=8, mew=2, label='Dipole source')

ax.set_xlim(X0 - 0.4, X1 + 0.4)
ax.set_ylim(Y0 - 0.4, Y1 + 0.4)
ax.set_aspect('equal')
ax.set_xlabel('x (µm)')
ax.set_ylabel('y (µm)')
ax.set_title('Photonic Crystal Nanocavity – Top View')

legend_handles = [
    mpatches.Patch(fc=BLUE, ec='none', alpha=0.85, label='Diamond slab'),
    mpatches.Patch(fc=WHITE, ec=BLACK, lw=0.4, label='Air holes'),
    plt.Line2D([0], [0], marker='x', color=RED, linestyle='None',
               markersize=8, markeredgewidth=2, label='Dipole source'),
]
fig.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(0.80, 0.5),
           fontsize=9, frameon=True, borderaxespad=0.0)
fig.tight_layout(rect=(0, 0, 0.78, 1))
plt.show()

<a id='scout'></a>
## 4. Stage 1 – Broadband Resonance Search

A **y-polarised point dipole** placed at the cavity centre drives a Gaussian pulse with a 12 % relative bandwidth, spanning ~580–700 nm. The only monitor is a zero-size **point probe** recording the time-domain field $E_y(t)$ throughout the simulation.

The simulation domain is enclosed by **PML (perfectly matched layer) boundaries** on all six sides to absorb outgoing radiation without reflections.

In [ ]:
def print_simulation_summary(sim, stage_label, wavelength_nm, bandwidth_rel, output_note):
    """Print a compact simulation summary without dumping the full object."""
    grid = sim.grid.num_cells
    total_cells = int(np.prod(grid))
    print(f'── {stage_label} ─────────────────────────────────────────────────')
    print(f'  Domain     : {sim.size[0]:.3f} × {sim.size[1]:.3f} × {sim.size[2]:.3f} µm')
    print(f'  Grid cells : {grid[0]} × {grid[1]} × {grid[2]} = {total_cells:,}')
    print(f'  Run time   : {sim.run_time * 1e12:.1f} ps')
    print(f'  Excitation : Ey dipole @ {wavelength_nm:.1f} nm, Δf/f = {bandwidth_rel * 100:.1f}%')
    print(f'  Outputs    : {output_note}')


def build_scout_simulation(structures, cx, cy, x0, x1, y0, y1,
                           thickness_um, wavelength_um,
                           bandwidth_rel, run_time_ps, min_steps,
                           x_trim=5.0, pml_pad=2.0, pml_layers=12):
    """
    Build the broadband scout simulation for resonance detection.

    The simulation domain covers the central portion of the nanobeam
    (trimming x_trim µm from each long-axis end) plus pml_pad µm of
    PML buffer on all sides.

    Parameters
    ----------
    structures : list[td.Structure]
    cx, cy : float  — cavity centre [µm]  (usually 0, 0 if GDS is centred)
    x0, x1, y0, y1 : float  — full GDS bounding box [µm]
    thickness_um, wavelength_um, bandwidth_rel, run_time_ps, min_steps
    x_trim : float  — µm to remove from each x-end
    pml_pad : float  — µm PML buffer on y and z faces
    pml_layers : int  — number of PML absorption layers

    Returns
    -------
    td.Simulation
    """
    f0     = C0 / (wavelength_um * 1e-6)
    fwidth = f0 * bandwidth_rel

    source = td.PointDipole(
        center=(cx, cy, 0.0),
        source_time=td.GaussianPulse(freq0=f0, fwidth=fwidth),
        polarization='Ey',
    )

    probe = td.FieldTimeMonitor(
        center=(cx, cy, 0.0),
        size=(0.0, 0.0, 0.0),
        name='probe',
        interval=5,
    )

    # Domain: trim x ends, pad y/z with PML buffer
    sx = (x1 - x0) - 2 * x_trim
    sy = (y1 - y0) + 2 * pml_pad
    sz = thickness_um + 2 * pml_pad

    pml = td.PML(num_layers=pml_layers)
    return td.Simulation(
        size=(sx, sy, sz),
        center=(cx, cy, 0.0),
        structures=structures,
        sources=[source],
        monitors=[probe],
        run_time=run_time_ps * 1e-12,
        grid_spec=td.GridSpec.auto(
            min_steps_per_wvl=min_steps,
            wavelength=wavelength_um,
        ),
        boundary_spec=td.BoundarySpec.all_sides(boundary=pml),
    )


sim_scout = build_scout_simulation(
    structures, CX, CY, X0, X1, Y0, Y1,
    THICKNESS_UM, SCOUT_WAVELENGTH_UM,
    SCOUT_BANDWIDTH, SCOUT_RUN_TIME_PS, MIN_STEPS_PER_WVL,
    x_trim=X_TRIM_UM, pml_pad=PML_PAD_UM, pml_layers=PML_LAYERS,
)

print_simulation_summary(
    sim_scout,
    stage_label='Scout Simulation',
    wavelength_nm=SCOUT_WAVELENGTH_UM * 1e3,
    bandwidth_rel=SCOUT_BANDWIDTH,
    output_note='probe ringdown monitor',
)

In [ ]:
if RUN_SIMULATIONS:
    import tidy3d.web as web
    data_scout = web.run(
        sim_scout,
        task_name=SCOUT_TASK_NAME,
        path=str(SCOUT_RUN_RESULTS),
    )
    print(f'Scout simulation complete: {SCOUT_RUN_RESULTS.name}')
else:
    print(f'Loaded scout cache: {SCOUT_LOAD_RESULTS.name}')
    data_scout = td.SimulationData.from_file(str(SCOUT_LOAD_RESULTS))
    print('  Probe time trace ready for resonance extraction.')

<a id='qfactor'></a>
## 5. Q-Factor Extraction

The Tidy3D **`ResonanceFinder`** plugin decomposes the time-domain ringdown $E_y(t)$ into a sum of decaying sinusoids using a matrix-pencil (Prony) method:

$$E_y(t) \approx \sum_k A_k \, e^{-(\omega_k / 2Q_k)\,t} \cos(\omega_k t + \phi_k)$$

Each mode is characterised by its frequency $\omega_k$ and quality factor $Q_k = \omega_k \tau_k / 2$, where $\tau_k$ is the energy decay time.

In [ ]:
def extract_resonance(data_scout, monitor_name='probe',
                      wavelength_centre_um=None, bandwidth_rel=None,
                      q_min=1e3, q_max=1e6,
                      init_num_freqs=400, rcond=1e-5):
    """
    Extract the dominant cavity resonance from the scout ringdown.

    Uses the Tidy3D ResonanceFinder (matrix-pencil / Prony method) via
    run_raw_signal() on the Ey time trace.  freq_window is computed
    automatically from the broadband source parameters.

    Parameters
    ----------
    data_scout : td.SimulationData
    monitor_name : str
    wavelength_centre_um : float  -- source centre wavelength [um]; defaults to SCOUT_WAVELENGTH_UM
    bandwidth_rel : float  -- relative bandwidth; defaults to SCOUT_BANDWIDTH
    q_min, q_max : float  -- Q-factor acceptance window
    init_num_freqs, rcond : ResonanceFinder hyper-parameters

    Returns
    -------
    dict with keys: freq_Hz, wavelength_um, Q, decay_time_ps, amplitude
    """
    if wavelength_centre_um is None:
        wavelength_centre_um = SCOUT_WAVELENGTH_UM
    if bandwidth_rel is None:
        bandwidth_rel = SCOUT_BANDWIDTH

    mon = data_scout[monitor_name]
    ey  = mon.Ey.values.squeeze().astype(complex)
    t   = mon.Ey.coords['t'].values
    dt  = float(t[1] - t[0])               # time step [s]

    # Frequency search window: source centre +/- 1 bandwidth
    f0     = C0 / (wavelength_centre_um * 1e-6)
    fwidth = f0 * bandwidth_rel
    freq_window = (f0 - fwidth, f0 + fwidth)

    rf  = ResonanceFinder(
        freq_window=freq_window,
        init_num_freqs=init_num_freqs,
        rcond=rcond,
    )
    # run_raw_signal takes a complex 1-D signal and the time step
    res = rf.run_raw_signal(signal=ey, time_step=dt)
    df  = res.to_dataframe()   # index = freq [Hz]; cols: decay, Q, amplitude, phase, error

    df = df[(df.index > 0) & (df['Q'] > q_min) & (df['Q'] < q_max)]
    df = df.sort_values('amplitude', ascending=False)

    if df.empty:
        raise RuntimeError(
            'ResonanceFinder found no valid modes. '
            'Try adjusting q_min or the simulation bandwidth.'
        )

    freq   = float(df.index[0])
    Q      = float(df['Q'].iloc[0])
    tau_ps = Q / (np.pi * freq) * 1e12
    wl_um  = (C0 / freq) * 1e6

    return {
        'freq_Hz':       freq,
        'wavelength_um': wl_um,
        'Q':             Q,
        'decay_time_ps': tau_ps,
        'amplitude':     float(df['amplitude'].iloc[0]),
    }


print('-- Q-factor extraction --')
q_results = extract_resonance(data_scout)
wavelength_lockin = q_results['wavelength_um']
print(f"  Resonance wavelength : {wavelength_lockin * 1e3:.3f} nm")
print(f"  Quality factor  Q    : {q_results['Q']:.0f}")
print(f"  Energy decay time tau  : {q_results['decay_time_ps']:.2f} ps")

# -- Ringdown visualisation
probe = data_scout['probe']
ey = probe.Ey.values.squeeze()
t  = probe.Ey.coords['t'].values * 1e12
env = np.abs(hilbert(ey))

dt        = np.diff(t[:2])[0] * 1e-12
freqs_fft = np.fft.rfftfreq(len(ey), d=dt)
spec      = np.abs(np.fft.rfft(ey))
wl_spec   = np.where(freqs_fft[1:] > 0, C0 / freqs_fft[1:] * 1e9, np.nan)
mask      = (wl_spec > 550) & (wl_spec < 750)
wl_plot   = wl_spec[mask] if np.any(mask) else wl_spec
spec_plot = spec[1:][mask] if np.any(mask) else spec[1:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

signal_abs     = np.abs(ey) / np.max(np.abs(ey))
env_n          = env / env.max()
ringdown_floor = 1e-6

ax1.semilogy(t, np.clip(signal_abs, ringdown_floor, None),
             color=BLUE, lw=0.9, alpha=0.55, label='$|E_y(t)|$')
ax1.semilogy(t, np.clip(env_n, ringdown_floor, None),
             color=RED, lw=1.8, label='Envelope |E|')
ax1.set_xlabel('Time (ps)')
ax1.set_ylabel('Normalised amplitude (log scale)')
ax1.set_title('Time-Domain Ringdown')
ax1.set_ylim(ringdown_floor, 1.2)
ax1.grid(True, which='both', alpha=0.25)
ax1.legend()

ax2.plot(wl_plot, spec_plot / spec_plot.max(), color=RED, lw=1.8)
ax2.axvline(wavelength_lockin * 1e3, color=BLUE, ls='--', lw=1.8,
            label=f"$\\lambda_0$ = {wavelength_lockin * 1e3:.2f} nm\n"
                  f"Q = {q_results['Q']:.0f}")
ax2.set_xlabel('Wavelength (nm)')
ax2.set_ylabel('Normalised PSD')
ax2.set_title('Resonance Spectrum')
ax2.legend()

plt.tight_layout()
plt.show()

<a id='lockin'></a>
## 6. Stage 2 – Narrowband Lock-in Simulation

The lock-in stage uses the resonance wavelength detected in Stage 1 as its centre frequency with a narrow 2 % bandwidth.  This minimises spectral leakage and maximises the steady-state field amplitude at the resonance.  Five monitors are added:

| Monitor | Type | Purpose |
|---------|------|---------|
| `probe` | `FieldTimeMonitor` | Q verification from ringdown |
| `field_near` | `FieldMonitor` (2-D, z = 0) | Near-field mode profile |
| `fld_3d_box` | `FieldMonitor` (3-D volume) | Mode volume integration |
| `farfield_angles` | `FieldProjectionAngleMonitor` | Angular radiation pattern |
| `farfield_kspace` | `FieldProjectionKSpaceMonitor` | k-space back-focal-plane map |

In [ ]:
def build_lockin_simulation(structures, cx, cy, x0, x1, y0, y1,
                            thickness_um, wavelength_um,
                            bandwidth_rel, run_time_ps, min_steps,
                            x_trim=5.0, pml_pad=2.0, pml_layers=12):
    """
    Build the narrowband lock-in simulation with the full monitor suite.

    Monitor placement uses the same domain trimming as the scout simulation.
    Far-field monitors sit 1.5 µm above the top slab surface.
    """
    f0     = C0 / (wavelength_um * 1e-6)
    fwidth = f0 * bandwidth_rel

    source = td.PointDipole(
        center=(cx, cy, 0.0),
        source_time=td.GaussianPulse(freq0=f0, fwidth=fwidth),
        polarization='Ey',
    )

    sx = (x1 - x0) - 2 * x_trim
    sy = (y1 - y0) + 2 * pml_pad
    sz = thickness_um + 2 * pml_pad

    # -- Time probe -----------------------------------------------------------
    probe = td.FieldTimeMonitor(
        center=(cx, cy, 0.0), size=(0, 0, 0), name='probe', interval=5,
    )

    # -- In-plane flux (80 % of domain) ---------------------------------------
    flux = td.FluxMonitor(
        center=(cx, cy, 0.0),
        size=(sx * 0.8, sy * 0.8, 0.0),
        freqs=[f0], name='flux',
    )

    # -- 2-D near-field (60 % of domain) --------------------------------------
    field_near = td.FieldMonitor(
        center=(cx, cy, 0.0),
        size=(sx * 0.6, sy * 0.6, 0.0),
        freqs=[f0], name='field_near',
    )

    # -- 3-D field box for mode volume (fixed physical extent) ----------------
    fld_3d = td.FieldMonitor(
        center=(cx, cy, 0.0),
        size=(6.0, 3.0, 2.0),
        freqs=[f0], name='fld_3d_box',
        fields=['Ex', 'Ey', 'Ez'],
    )

    # -- Far-field projections 1.5 µm above slab surface ----------------------
    z_ff    = thickness_um / 2 + 1.5
    ff_size = (8.0, 8.0, 0.0)

    farfield_cartesian = td.FieldProjectionCartesianMonitor(
        center=(cx, cy, z_ff), size=ff_size, freqs=[f0],
        name='farfield_cartesian',
        x=np.linspace(-4, 4, 50),
        y=np.linspace(-4, 4, 50),
        proj_axis=2,
    )

    farfield_kspace = td.FieldProjectionKSpaceMonitor(
        center=(cx, cy, z_ff), size=ff_size, freqs=[f0],
        name='farfield_kspace',
        ux=np.linspace(-0.95, 0.95, 60),
        uy=np.linspace(-0.95, 0.95, 60),
        proj_axis=2,
    )

    theta = np.deg2rad(np.linspace(0.0, 90.0, 100))
    phi   = np.deg2rad(np.linspace(0.0, 360.0, 200))
    farfield_angles = td.FieldProjectionAngleMonitor(
        center=(cx, cy, z_ff), size=ff_size, freqs=[f0],
        name='farfield_angles',
        theta=theta, phi=phi, normal_dir='+',
    )

    pml = td.PML(num_layers=pml_layers)
    return td.Simulation(
        size=(sx, sy, sz),
        center=(cx, cy, 0.0),
        structures=structures,
        sources=[source],
        monitors=[probe, flux, field_near, fld_3d,
                  farfield_cartesian, farfield_kspace, farfield_angles],
        run_time=run_time_ps * 1e-12,
        grid_spec=td.GridSpec.auto(
            min_steps_per_wvl=min_steps,
            wavelength=wavelength_um,
        ),
        boundary_spec=td.BoundarySpec.all_sides(boundary=pml),
    )


sim_lockin = build_lockin_simulation(
    structures, CX, CY, X0, X1, Y0, Y1,
    THICKNESS_UM, wavelength_lockin,
    LOCKIN_BANDWIDTH, LOCKIN_RUN_TIME_PS, MIN_STEPS_PER_WVL,
    x_trim=X_TRIM_UM, pml_pad=PML_PAD_UM, pml_layers=PML_LAYERS,
)

print_simulation_summary(
    sim_lockin,
    stage_label='Lock-in Simulation',
    wavelength_nm=wavelength_lockin * 1e3,
    bandwidth_rel=LOCKIN_BANDWIDTH,
    output_note='probe, flux, near field, 3-D box, far-field projections',
)

In [ ]:
if RUN_SIMULATIONS and RUN_LOCKIN_STAGE:
    import tidy3d.web as web
    data = web.run(
        sim_lockin,
        task_name=LOCKIN_TASK_NAME,
        path=str(LOCKIN_RUN_RESULTS),
    )
    print(f'Lock-in simulation complete: {LOCKIN_RUN_RESULTS.name}')
else:
    if RUN_SIMULATIONS and not RUN_LOCKIN_STAGE:
        print(
            f'{SIMULATION_PROFILE} selected: skipping Stage 2 rerun and '
            f'reusing {LOCKIN_LOAD_RESULTS.name}.'
        )
    else:
        print(f'Loaded lock-in cache: {LOCKIN_LOAD_RESULTS.name}')
    data = td.SimulationData.from_file(str(LOCKIN_LOAD_RESULTS))
    print(f"  Monitor suite ready: {len(data.monitor_data)} datasets loaded.")

<a id='nearfield'></a>
## 7. Near-Field Mode Analysis

The `field_near` monitor records the electric field components $E_x, E_y, E_z$ on a 2-D cross-section at $z = 0$ (mid-plane of the slab).  We visualise the total intensity $|\mathbf{E}|^2 = |E_x|^2 + |E_y|^2 + |E_z|^2$, overlay the fabricated GDS outline, and characterise the lateral confinement by fitting 1-D Gaussian profiles.

The **effective mode area** is defined as:

$$A_{\text{eff}} = \frac{\left(\iint I \, dA\right)^2}{\iint I^2 \, dA}$$

In [ ]:
def _coords_to_um(arr):
    """Auto-convert coordinate array from metres to µm if needed."""
    return arr * 1e6 if np.max(np.abs(arr)) < 1e-3 else arr


def extract_nearfield_intensity(data, monitor_name='field_near'):
    """Return 2-D intensity map and coordinate arrays (in µm)."""
    mon = data[monitor_name]
    Ex  = mon.Ex.isel(f=0).values.squeeze()
    Ey  = mon.Ey.isel(f=0).values.squeeze()
    Ez  = mon.Ez.isel(f=0).values.squeeze() if 'Ez' in mon else 0.0
    I   = np.abs(Ex) ** 2 + np.abs(Ey) ** 2 + np.abs(Ez) ** 2
    x   = _coords_to_um(mon.Ey.coords['x'].values)
    y   = _coords_to_um(mon.Ey.coords['y'].values)
    # Ensure I has shape (ny, nx) to match pcolormesh convention
    if I.shape == (len(x), len(y)):
        I = I.T
    return I, x, y


def crop_field(I, x, y, x_lim=(-2.0, 2.0), y_lim=(-0.7, 0.7)):
    """Crop a 2-D intensity map to the specified window (µm)."""
    xi = (x >= x_lim[0]) & (x <= x_lim[1])
    yi = (y >= y_lim[0]) & (y <= y_lim[1])
    return I[np.ix_(yi, xi)], x[xi], y[yi]


def overlay_gds_outline(ax, x_window, y_window, holes_gds=HOLES_GDS):
    """Overlay the fabricated GDS outline on top of the near-field map."""
    rect_kw_outer = dict(fill=False, ec=BLACK, lw=1.4, alpha=0.55, zorder=4)
    rect_kw_inner = dict(fill=False, ec=WHITE, lw=0.8, alpha=0.95, zorder=5)
    ax.add_patch(plt.Rectangle((X0, Y0), X1 - X0, Y1 - Y0, **rect_kw_outer))
    ax.add_patch(plt.Rectangle((X0, Y0), X1 - X0, Y1 - Y0, **rect_kw_inner))

    lib = gdstk.read_gds(str(holes_gds))
    scale = lib.unit / 1e-6
    cell = lib.top_level()[0]

    for poly in cell.polygons:
        if poly.layer != 0 or poly.datatype != 0:
            continue
        pts = poly.points * scale
        if (
            pts[:, 0].max() < x_window[0] or pts[:, 0].min() > x_window[1]
            or pts[:, 1].max() < y_window[0] or pts[:, 1].min() > y_window[1]
        ):
            continue
        ax.add_patch(plt.Polygon(
            pts, closed=True, fill=False, ec=BLACK, lw=1.0,
            alpha=0.5, zorder=4, joinstyle='round'
        ))
        ax.add_patch(plt.Polygon(
            pts, closed=True, fill=False, ec=WHITE, lw=0.4,
            alpha=0.95, zorder=5, joinstyle='round'
        ))


def plot_nearfield(I, x, y, wavelength_nm, cmap=None):
    """Display the near-field intensity map."""
    cmap = cmap or mono_cmap
    I_n  = I / I.max()
    vmin, vmax = np.percentile(I_n, [0.1, 99.9])

    fig, ax = plt.subplots(figsize=(8.2, 3.6))
    im = ax.pcolormesh(x, y, I_n, cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    overlay_gds_outline(ax, (x.min(), x.max()), (y.min(), y.max()))
    cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label('$|\\mathbf{E}|^2$ (normalised)')
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(y.min(), y.max())
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_title(f'Near-Field Intensity  –  $\\lambda_0$ = {wavelength_nm:.2f} nm')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()


def gaussian_1d(x, A, x0, sigma, bg):
    return A * np.exp(-2 * (x - x0) ** 2 / sigma ** 2) + bg


def compute_confinement(I, x, y, wavelength_um):
    """
    Quantify the lateral confinement from 1-D Gaussian fits.

    Returns a dict with 1/e² widths, FWHM, effective area, and
    effective area normalised to λ².
    """
    results = {}
    for axis_label, coords, profile in [
        ('x', x, I.sum(axis=0)),
        ('y', y, I.sum(axis=1)),
    ]:
        pn = profile / profile.max()
        try:
            p0   = [1.0, coords[np.argmax(pn)], 0.25, 0.0]
            popt, _ = curve_fit(gaussian_1d, coords, pn, p0=p0, maxfev=8000)
            w = abs(popt[2])      # 1/e² half-width
        except RuntimeError:
            above = coords[pn > np.exp(-2)]
            w = (above[-1] - above[0]) / 2 if len(above) > 1 else np.nan
        results[f'w_{axis_label}_um']    = w
        results[f'fwhm_{axis_label}_nm'] = 2 * np.sqrt(2 * np.log(2)) * w * 1e3

    dx     = abs(np.diff(x[:2])[0])
    dy     = abs(np.diff(y[:2])[0])
    A_eff  = float(np.sum(I) ** 2 / np.sum(I ** 2) * dx * dy)
    results['A_eff_um2']    = A_eff
    results['A_eff_lambda2'] = A_eff / wavelength_um ** 2
    return results


# ── Run analysis ──────────────────────────────────────────────────────────────
print('── Near-field analysis ──────────────────────────────────────────────────')

I_raw, x_raw, y_raw = extract_nearfield_intensity(data)
I_nf, x_nf, y_nf   = crop_field(I_raw, x_raw, y_raw)
plot_nearfield(I_nf, x_nf, y_nf, wavelength_lockin * 1e3)

conf = compute_confinement(I_nf, x_nf, y_nf, wavelength_lockin)
print(f"  1/e² width  w_x   = {conf['w_x_um'] * 1e3:.0f} nm"
      f"  (FWHM = {conf['fwhm_x_nm']:.0f} nm)")
print(f"  1/e² width  w_y   = {conf['w_y_um'] * 1e3:.0f} nm"
      f"  (FWHM = {conf['fwhm_y_nm']:.0f} nm)")
print(f"  Eff. mode area    = {conf['A_eff_um2']:.4f} µm²"
      f"  = {conf['A_eff_lambda2']:.3f} λ²")

<a id='modevolume'></a>
## 8. Mode Volume and Purcell Enhancement

The **effective mode volume** is computed from the 3-D field distribution stored in `fld_3d_box`:

$$V_{\text{eff}} = \frac{\int \varepsilon(\mathbf{r})\,|\mathbf{E}(\mathbf{r})|^2 \, d^3r}{\max\!\left[\varepsilon(\mathbf{r})\,|\mathbf{E}(\mathbf{r})|^2\right]}$$

The permittivity $\varepsilon(\mathbf{r})$ is taken as $n_{\text{diamond}}^2$ inside the slab and 1 (air) outside.  The Purcell factor follows directly from $Q$ and $V_{\text{eff}}$.

In [ ]:
def compute_mode_volume(data, monitor_name='fld_3d_box',
                        wavelength_um=0.62, thickness_um=0.136):
    """
    Compute the effective mode volume V_eff and its normalisation (λ/2n)³.

    The permittivity is estimated analytically: n_diamond² inside the slab
    (|z| < thickness_um/2) and 1 elsewhere.

    Parameters
    ----------
    data : td.SimulationData
    monitor_name : str
    wavelength_um : float  — resonance wavelength [µm]
    thickness_um : float   — slab thickness [µm]

    Returns
    -------
    V_eff_um3 : float  — effective mode volume [µm³]
    V_norm    : float  — V_eff / (λ/2n)³  (dimensionless)
    """
    mon = data[monitor_name]
    Ex  = mon.Ex.isel(f=0).values.squeeze()
    Ey  = mon.Ey.isel(f=0).values.squeeze()
    Ez  = mon.Ez.isel(f=0).values.squeeze() if 'Ez' in mon else np.zeros_like(Ex)

    x   = _coords_to_um(mon.Ex.coords['x'].values)
    y   = _coords_to_um(mon.Ex.coords['y'].values)
    z   = _coords_to_um(mon.Ex.coords['z'].values)

    # Ensure field shape (nz, ny, nx)
    I = np.abs(Ex) ** 2 + np.abs(Ey) ** 2 + np.abs(Ez) ** 2
    if I.shape == (len(x), len(y), len(z)):
        I = np.transpose(I, (2, 1, 0))

    # Permittivity map
    n   = float(n_diamond(wavelength_um))
    Z3d = z[:, None, None] * np.ones_like(I)
    eps = np.where(np.abs(Z3d) < thickness_um / 2, n ** 2, 1.0)

    eps_I = eps * I
    dV    = abs(np.diff(x[:2])[0]) * abs(np.diff(y[:2])[0]) * abs(np.diff(z[:2])[0])
    V_eff = float(np.sum(eps_I) * dV / np.max(eps_I))

    lambda_n = wavelength_um / n
    V_norm   = V_eff / (lambda_n / 2) ** 3
    return V_eff, V_norm


def compute_purcell(Q, V_eff_um3, wavelength_um):
    """
    Purcell enhancement factor  F_P = (3/4π²) (λ/n)³ Q / V_eff.

    Parameters
    ----------
    Q : float  — quality factor
    V_eff_um3 : float  — effective mode volume [µm³]
    wavelength_um : float  — resonance wavelength [µm]

    Returns
    -------
    float  — Purcell factor
    """
    n       = float(n_diamond(wavelength_um))
    lam_n3  = (wavelength_um / n) ** 3    # (λ/n)³ [µm³]
    return float((3.0 / (4.0 * np.pi ** 2)) * lam_n3 / V_eff_um3 * Q)


print('── Mode volume and Purcell factor ───────────────────────────────────────')
V_eff, V_norm = compute_mode_volume(data,
                                    wavelength_um=wavelength_lockin,
                                    thickness_um=THICKNESS_UM)
Q_val = q_results['Q']
F_P   = compute_purcell(Q_val, V_eff, wavelength_lockin)

print(f'  Effective mode volume  V_eff  = {V_eff:.4f} µm³')
print(f'                                = {V_norm:.3f} × (λ/2n)³')
print(f'  Quality factor         Q      = {Q_val:.0f}')
print(f'  Purcell factor         F_P    = {F_P:.1f}')

<a id='farfield'></a>
## 9. Far-Field Radiation Pattern and Collection Efficiency

The `farfield_kspace` monitor projects the fields onto a plane at height $z_{\text{ff}}$ and records the complex amplitudes as a function of normalised wavevectors $(u_x, u_y) = (k_x/k_0, k_y/k_0)$.  A circle of radius NA in this plane corresponds to the acceptance cone of an objective lens.

Collection efficiency is:

$$\eta_{\text{NA}} = \frac{\displaystyle\iint_{u_x^2 + u_y^2 \leq \text{NA}^2} I(u_x, u_y)\, du_x\, du_y}{\displaystyle\iint_{u_x^2 + u_y^2 \leq 1} I(u_x, u_y)\, du_x\, du_y}$$

In [ ]:
def extract_kspace_intensity(data, monitor_name='farfield_kspace'):
    """Return far-field intensity on the (ux, uy) k-space grid."""
    mon = data[monitor_name]
    # Get field projections – try Etheta/Ephi first, fall back to Ex/Ey
    if hasattr(mon, 'Etheta'):
        dims = mon.Etheta.dims
        if 'r' in dims:
            Et = mon.Etheta.isel(f=0, r=0).values.squeeze()
            Ep = mon.Ephi.isel(f=0, r=0).values.squeeze()
        else:
            Et = mon.Etheta.isel(f=0).values.squeeze()
            Ep = mon.Ephi.isel(f=0).values.squeeze()
    else:
        Et = mon.Ex.isel(f=0).values.squeeze()
        Ep = mon.Ey.isel(f=0).values.squeeze()
    I  = np.abs(Et) ** 2 + np.abs(Ep) ** 2
    ux = np.asarray(mon.ux)
    uy = np.asarray(mon.uy)
    return I, ux, uy


def collection_efficiency(I, ux, uy, na, n_bg=1.0):
    """
    Fraction of upper-hemisphere power collected within NA.

    Parameters
    ----------
    I : 2-D array  — intensity on (ux, uy) grid  (shape: n_ux × n_uy)
    ux, uy : 1-D arrays
    na, n_bg : float

    Returns
    -------
    float  — collection efficiency [0, 1]
    """
    UX, UY = np.meshgrid(ux, uy, indexing='ij')
    R2     = UX ** 2 + UY ** 2
    inside_na  = R2 <= (na / n_bg) ** 2
    inside_all = R2 <= 1.0
    total = np.sum(I[inside_all])
    if total == 0:
        return 0.0
    return float(np.sum(I[inside_na]) / total)


# ── Compute and plot ───────────────────────────────────────────────────────────
print('── Far-field radiation pattern ──────────────────────────────────────────')

I_ff, ux, uy = extract_kspace_intensity(data)
eta_NA = collection_efficiency(I_ff, ux, uy, NA, N_BG)
print(f'  Collection efficiency at NA = {NA}: {eta_NA * 100:.1f}%')

# NA sweep
na_sweep  = np.linspace(0.05, 0.99, 80)
eta_sweep = [collection_efficiency(I_ff, ux, uy, na_) for na_ in na_sweep]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.2))

# k-space map
UX_g, UY_g = np.meshgrid(ux, uy, indexing='ij')
ax1.pcolormesh(ux, uy, I_ff.T / I_ff.max(), cmap=mono_cmap,
               vmin=0, vmax=1, shading='auto')
theta_c = np.linspace(0, 2 * np.pi, 300)
for na_c, ls_c, lbl_c in [(NA, '--', f'NA = {NA:.2f}'), (0.9, ':', 'NA = 0.90')]:
    if na_c <= 1.0:
        ax1.plot(na_c * np.cos(theta_c), na_c * np.sin(theta_c),
                 color=WHITE, lw=1.7, ls=ls_c, alpha=0.95)
ax1.set_aspect('equal')
ax1.set_xlabel('$u_x = k_x/k_0$')
ax1.set_ylabel('$u_y = k_y/k_0$')
ax1.set_title('Far-Field Radiation Pattern (k-space)')
ax1.text(0.97, 0.05, f'--  NA = {NA:.2f}\n:   NA = 0.90',
         transform=ax1.transAxes, ha='right', va='bottom', color=WHITE,
         fontsize=9, bbox=dict(boxstyle='round,pad=0.28',
         fc=(0, 0, 0, 0.35), ec='none'))

# Collection efficiency curve
ax2.plot(na_sweep, np.array(eta_sweep) * 100, color=RED, lw=2)
ax2.axvline(NA, color=BLUE, ls='--', lw=1.8,
            label=f'NA = {NA}  →  {eta_NA * 100:.1f}%')
ax2.set_xlabel('Numerical aperture (NA)')
ax2.set_ylabel('Collection efficiency (%)')
ax2.set_title('Collection Efficiency vs NA')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.show()

<a id='polarisation'></a>
## 10. Polarisation Analysis

The polarisation state of the emitted light is characterised by the **Stokes parameters**, derived from the spherical far-field components $(E_\theta, E_\phi)$:

$$S_0 = |E_\theta|^2 + |E_\phi|^2, \quad
S_1 = |E_\theta|^2 - |E_\phi|^2, \quad
S_2 = 2\,\text{Re}(E_\theta E_\phi^*), \quad
S_3 = 2\,\text{Im}(E_\theta E_\phi^*)$$

Key metrics:

| Metric | Formula | Meaning |
|--------|---------|--------|
| DoLP | $\sqrt{S_1^2 + S_2^2} / S_0$ | Degree of linear polarisation |
| DoCP | $|S_3| / S_0$ | Degree of circular polarisation |
| DoP | $\sqrt{S_1^2 + S_2^2 + S_3^2} / S_0$ | Total degree of polarisation |
| $\psi$ | $\frac{1}{2}\arctan(S_2/S_1)$ | Orientation angle of the polarisation ellipse |

In [ ]:
def compute_stokes(data, monitor_name='farfield_angles',
                   na=0.65, n_bg=1.0):
    """
    Compute Stokes parameters and polarisation metrics from the far-field.

    Cartesian field components (Ex, Ey, Ez) are projected onto the spherical
    basis (E_theta, E_phi) using the standard physics convention for the
    e_theta and e_phi unit vectors.  Results are returned per (theta, phi)
    angle pair and as NA-weighted scalar averages.

    Parameters
    ----------
    data : td.SimulationData
    monitor_name : str
    na : float  — collection NA for weighting
    n_bg : float

    Returns
    -------
    dict  — contains S0..S3, DoLP, DoCP, DoP, psi, chi,
             theta/phi arrays, NA_mask, and averaged scalars
    """
    mon = data[monitor_name]

    theta = np.asarray(mon.theta)   # shape (n_theta,) [rad]
    phi   = np.asarray(mon.phi)     # shape (n_phi,)   [rad]

    # Extract or compute spherical field components
    if hasattr(mon, 'Etheta'):
        # Already in spherical basis
        dims = mon.Etheta.dims
        if 'r' in dims:
            Et = mon.Etheta.isel(f=0, r=0).values
            Ep = mon.Ephi.isel(f=0, r=0).values
        else:
            Et = mon.Etheta.isel(f=0).values
            Ep = mon.Ephi.isel(f=0).values
    else:
        # Project Cartesian → spherical
        Ex = mon.Ex.isel(f=0).values.squeeze()
        Ey = mon.Ey.isel(f=0).values.squeeze()
        Ez = mon.Ez.isel(f=0).values.squeeze() if hasattr(mon, 'Ez') else 0.0
        TH, PH = np.meshgrid(theta, phi, indexing='ij')
        ct, st = np.cos(TH), np.sin(TH)
        cp, sp = np.cos(PH), np.sin(PH)
        Et = ct * cp * Ex + ct * sp * Ey - st * Ez
        Ep = -sp * Ex + cp * Ey

    eps = 1e-30
    S0  = np.abs(Et) ** 2 + np.abs(Ep) ** 2
    S1  = np.abs(Et) ** 2 - np.abs(Ep) ** 2
    S2  = 2 * np.real(Et * np.conj(Ep))
    S3  = 2 * np.imag(Et * np.conj(Ep))

    DoLP = np.sqrt(S1 ** 2 + S2 ** 2) / (S0 + eps)
    DoCP = np.abs(S3)               / (S0 + eps)
    DoP  = np.sqrt(S1**2 + S2**2 + S3**2) / (S0 + eps)
    psi  = 0.5 * np.arctan2(S2, S1)
    chi  = 0.5 * np.arcsin(np.clip(S3 / (S0 + eps), -1, 1))

    # NA mask and S0-weighted averages within collection cone
    TH_g, _ = np.meshgrid(theta, phi, indexing='ij')
    theta_max = np.arcsin(min(na / n_bg, 0.9999))
    NA_mask   = TH_g <= theta_max
    w         = np.sin(TH_g)             # solid-angle weight sin(θ)

    def wavg(arr):
        wt = (S0 * w * NA_mask).sum() + eps
        return float((arr * S0 * w * NA_mask).sum() / wt)

    return {
        'S0': S0, 'S1': S1, 'S2': S2, 'S3': S3,
        'DoLP': DoLP, 'DoCP': DoCP, 'DoP': DoP,
        'psi': psi, 'chi': chi,
        'theta': theta, 'phi': phi, 'NA_mask': NA_mask,
        'DoLP_avg': wavg(DoLP),
        'DoCP_avg': wavg(DoCP),
        'DoP_avg':  wavg(DoP),
        'psi_avg':  wavg(psi),
        'theta_max_deg': np.rad2deg(theta_max),
    }


def plot_stokes(pol, na):
    """Plot S0 intensity, DoLP, and DoCP on θ-φ heatmaps."""
    theta_deg = np.rad2deg(pol['theta'])
    phi_deg   = np.rad2deg(pol['phi'])

    panels = [
        ('$S_0$ (Intensity)', pol['S0'] / pol['S0'].max(), mono_cmap, 0, 1, 'Normalised $S_0$'),
        ('DoLP',             pol['DoLP'],                  mono_cmap, 0, 1, 'DoLP'),
        ('DoCP',             pol['DoCP'],                  mono_cmap, 0, 1, 'DoCP'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
    for ax, (title, arr, cmap, vmin, vmax, cbar_label) in zip(axes, panels):
        im = ax.pcolormesh(phi_deg, theta_deg, arr,
                           cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
        cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label(cbar_label)
        ax.axhline(pol['theta_max_deg'], color=WHITE, ls='--', lw=1.4, alpha=0.95)
        ax.set_xlabel('$\\phi$ (°)')
        ax.set_ylabel('$\\theta$ (°)')
        ax.set_title(title)
        ax.text(0.97, 0.05, f'NA = {na:.2f}', transform=ax.transAxes,
                ha='right', va='bottom', color=WHITE, fontsize=9,
                bbox=dict(boxstyle='round,pad=0.28', fc=(0, 0, 0, 0.35), ec='none'))
    plt.tight_layout()
    plt.show()


print('── Polarisation analysis ────────────────────────────────────────────────')
pol = compute_stokes(data, na=NA)

print(f"  DoLP  (linear)  = {pol['DoLP_avg']:.3f}")
print(f"  DoCP  (circular)= {pol['DoCP_avg']:.3f}")
print(f"  DoP   (total)   = {pol['DoP_avg']:.3f}")
print(f"  ψ (orientation) = {np.rad2deg(pol['psi_avg']):.1f}°")

plot_stokes(pol, NA)

<a id='summary'></a>
## 11. Results Summary

In [ ]:
w = 62  # column width
print('=' * w)
print('  DIAMOND PHOTONIC CRYSTAL NANOCAVITY – SIMULATION RESULTS')
print('=' * w)

print('\n  Structure')
print(f"    Slab thickness         : {THICKNESS_UM * 1e3:.1f} nm")
print(f"    Sidewall angle         : {SIDEWALL_ANGLE_DEG}°")
print(f"    Diamond index at λ₀    : {float(n_diamond(wavelength_lockin)):.4f}")

print('\n  Resonance (from scout stage)')
print(f"    Resonance wavelength λ₀: {wavelength_lockin * 1e3:.3f} nm")
print(f"    Quality factor  Q      : {q_results['Q']:.0f}")
print(f"    Energy decay time τ    : {q_results['decay_time_ps']:.2f} ps")

print('\n  Near-Field Confinement')
print(f"    1/e² width w_x         : {conf['w_x_um'] * 1e3:.0f} nm")
print(f"    1/e² width w_y         : {conf['w_y_um'] * 1e3:.0f} nm")
print(f"    Eff. mode area A_eff   : {conf['A_eff_um2']:.4f} µm²"
      f" = {conf['A_eff_lambda2']:.3f} λ²")

print('\n  Mode Volume and Purcell Enhancement')
print(f"    Eff. mode volume V_eff : {V_eff:.4f} µm³ = {V_norm:.3f} × (λ/2n)³")
print(f"    Purcell factor  F_P    : {F_P:.1f}")

print(f'\n  Collection (NA = {NA})')
print(f"    Collection efficiency η: {eta_NA * 100:.1f}%")

print('\n  Far-Field Polarisation')
print(f"    DoLP (linear)          : {pol['DoLP_avg']:.3f}")
print(f"    DoCP (circular)        : {pol['DoCP_avg']:.3f}")
print(f"    DoP  (total)           : {pol['DoP_avg']:.3f}")
print(f"    Orientation angle ψ    : {np.rad2deg(pol['psi_avg']):.1f}°")

print('\n' + '=' * w)

## Conclusion

This notebook demonstrated a complete FDTD characterisation workflow for a diamond photonic crystal nanocavity using Tidy3D.  The two-stage approach (broadband scout + narrowband lock-in) efficiently combines resonance detection with detailed mode analysis at a fraction of the cost of a single long broadband simulation.

**Key take-aways:**

- The Tidy3D `ResonanceFinder` reliably extracts resonance wavelength and Q-factor from a ~12 ps ringdown without requiring the simulation to fully decay.
- Mode volume well below $(\lambda/2n)^3$ and a Purcell factor $F_P \gg 1$ confirm strong light–matter interaction, favouring efficient single-photon emission from SnV centres.
- The collection efficiency above 30–50 % (depending on NA) shows that a significant fraction of the emitted photons are directional and can be captured by a standard objective.
- The high degree of linear polarisation (DoLP) facilitates polarisation-selective collection and analysis of SnV-cavity emission.

---

## References

1. Faraon, A. *et al.* "Resonant enhancement of the zero-phonon emission from a colour centre in a diamond cavity." *Nature Photonics* **5**, 301–305 (2011). doi:10.1038/nphoton.2011.52.
2. Riedrich-Möller, J. *et al.* "One- and two-dimensional photonic crystal microcavities in single crystal diamond." *Nature Nanotechnology* **7**, 69–74 (2012). doi:10.1038/nnano.2011.190.
3. Zaitsev, A. M. *Optical Properties of Diamond: A Data Handbook.* Springer, Berlin, Heidelberg (2001). doi:10.1007/978-3-662-04548-0.
4. Almutlaq, J., Buzzi, A., Khaykin, A., Li, L., Yzaguirre, W., Sirotin, M., Gilbert, G., Clark, G., and Englund, D. "Foundry-Enabled Patterning of Diamond Quantum Microchiplets for Scalable Quantum Photonics." arXiv:2601.20025 [quant-ph] (2026). doi:10.48550/arXiv.2601.20025.
5. Flexcompute. *Tidy3D Documentation.* https://docs.flexcompute.com/projects/tidy3d/en/latest/ (accessed 2026-04-06).
6. Purcell, E. M. "Spontaneous emission probabilities at radio frequencies." In *Proceedings of the American Physical Society*, *Physical Review* **69**, 674 (1946). doi:10.1103/PhysRev.69.674.2.